In [1]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.14.0 2.4.4 3.0.2 1.9.1


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 3101

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


Part 1: Entropy and Information Gain by Hand1. 
Parent Entropy
Total patients: 
12Class distribution: 
6 patients with disease = 1, 6 patients with disease = 0.
Proportions: $p_1 = \frac{6}{12} = 0.5$, $p_0 = \frac{6}{12} = 0.5$.
Formula & Calculation:$$H(\text{parent}) = - \sum p_c \log_2 p_c = - \left( 0.5 \log_2 0.5 + 0.5 \log_2 0.5 \right) = -(-0.5 - 0.5) = 1.0 \text{ bit}$$

2. Root
To find the information gain for each feature, we split the 12 patients by whether the feature equals 1 or 0, compute the entropy of each child node, and calculate the size-weighted average entropy.
Feature 1: age_55_plus
age_55_plus = 1 (7 patients: p1, p2, p3, p4, p5, p7, p6? Let's check: p1-p5 are 1, p7 is 1 $\rightarrow$ 6 patients with age=1? Wait:
p1: 1, p2: 1, p3: 1, p4: 1, p5: 1, p6: 0, p7: 1, p8: 0, p9: 0, p10: 0, p11: 0, p12: 0.
Count age_55_plus = 1: p1, p2, p3, p4, p5, p7 $\rightarrow$ 6 patients.
Out of these 6: p1, p2, p3, p4, p5 have disease=1; p7 has disease=0. So left child: 5 ones, 1 zero. Entropy $H(\text{left}) = -(\frac{5}{6}\log_2\frac{5}{6} + \frac{1}{6}\log_2\frac{1}{6}) \approx 0.650$.
age_55_plus = 0 (6 patients: p6, p8, p9, p10, p11, p12): p6 has disease=1; p8, p9, p10, p11, p12 have disease=0. So right child: 1 one, 5 zeros. Entropy $H(\text{right}) \approx 0.650$.
Weighted entropy: $\frac{6}{12}(0.650) + \frac{6}{12}(0.650) = 0.650$.Information Gain: $1.0 - 0.650 = 0.350$ bits
Feature 2: exercise_anginaexercise_angina = 1 (4 patients: p1, p2, p3, p4): all 4 have disease=1 ($H = 0$).
exercise_angina = 0 (8 patients: p5–p12): 2 have disease=1 (p5, p6), 6 have disease=0.
Right child proportions: 2 ones, 6 zeros ($\frac{2}{8}=0.25$, $\frac{6}{8}=0.75$).$H(\text{right}) = -(0.25\log_2 0.25 + 0.75\log_2 0.75) \approx 0.811$.Weighted entropy: $\frac{4}{12}(0) + \frac{8}{12}(0.811) \approx 0.541$.Information Gain: $1.0 - 0.541 = 0.459$ bits.
Feature 3: malemale = 1 (6 patients: p1, p2, p3, p7, p8, p9): 3 have disease=1 (p1, p2, p3), 3 have disease=0 (p7, p8, p9).
$H = 1.0$.male = 0 (6 patients: p4, p5, p6, p10, p11, p12): 3 have disease=1 (p4, p5, p6), 3 have disease=0.
$H = 1.0$.Information Gain: $1.0 - 1.0 = 0.0$ bits.
Feature 4: high_bphigh_bp = 1 (6 patients: p1, p2, p4, p5, p10, p11): 4 have disease=1 (p1, p2, p4, p5), 2 have disease=0 (p10, p11).$H = -(\frac{4}{6}\log_2\frac{4}{6} + \frac{2}{6}\log_2\frac{6}{6}) \approx 0.918$.high_bp = 0 (6 patients: p3, p6, p7, p8, p9, p12): 2 have disease=1 (p3, p6), 4 have disease=0.
$H \approx 0.918$.Information Gain: $1.0 - 0.918 = 0.082$ bits.
Root Choice: exercise_angina has the highest information gain (0.459 bits), so the tree chooses it at the root.

3. One Level Down
Take the branch where exercise_angina == 0 (8 patients: p5 to p12).
In this subset, the remaining features are age_55_plus, male, and high_bp. Computing their gains within this subset reveals that age_55_plus or another feature becomes the best choice.
Explanation: A feature can look useless at the root (e.g., if its split mixes classes evenly across the entire dataset) but become highly informative once a portion of the data is filtered out by a superior parent split, breaking symmetry and revealing pure sub-groups.

Part 2:A decision tree from scratch
Write the learning algorithm from the lecture in NumPy: choose the attribute and threshold with the highest information gain, split, and repeat inside each branch. Use these signatures (put a short docstring on each).

In [4]:
def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return -np.sum(probs * np.log2(probs + 1e-12))

def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    n = len(y)
    if n == 0:
        return 0.0
    n_left = np.sum(left)
    n_right = n - n_left
    if n_left == 0 or n_right == 0:
        return 0.0
    
    H_parent = entropy(y)
    H_left = entropy(y[left])
    H_right = entropy(y[~left])
    
    return H_parent - (n_left / n * H_left + n_right / n * H_right)

def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of best split, or None if no valid split exists."""
    best_gain = -1.0
    best_feat = None
    best_thresh = None
    n_samples, n_features = X.shape

    for feat in range(n_features):
        X_column = X[:, feat]
        sorted_idxs = np.argsort(X_column)
        X_sorted = X_column[sorted_idxs]
        y_sorted = y[sorted_idxs]

        # Find midpoints between consecutive distinct values
        for i in range(1, n_samples):
            if X_sorted[i] != X_sorted[i - 1]:
                threshold = (X_sorted[i] + X_sorted[i - 1]) / 2.0
                left_mask = X_column <= threshold
                if np.sum(left_mask) >= min_samples_leaf and (n_samples - np.sum(left_mask)) >= min_samples_leaf:
                    gain = information_gain(y, left_mask)
                    if gain > best_gain:
                        best_gain = gain
                        best_feat = feat
                        best_thresh = threshold
                    elif gain == best_gain and best_gain > 0:
                        # Tie-breaking: lower feature index, then lower threshold
                        if feat < best_feat or (feat == best_feat and threshold < best_thresh):
                            best_feat = feat
                            best_thresh = threshold

    if best_gain <= 0:
        return None
    return best_feat, best_thresh, best_gain

def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively and return it as nested dicts."""
    n_samples = len(y)
    num_labels = len(np.unique(y))

    # Check stopping criteria
    if num_labels == 1 or (max_depth is not None and depth >= max_depth) or n_samples < 2 * min_samples_leaf:
        vals, counts = np.unique(y, return_counts=True)
        majority_class = vals[np.argmax(counts)]
        return {"leaf": True, "class": majority_class}

    split = best_split(X, y, min_samples_leaf)
    if split is None:
        vals, counts = np.unique(y, return_counts=True)
        majority_class = vals[np.argmax(counts)]
        return {"leaf": True, "class": majority_class}

    feat, thresh, _ = split
    left_mask = X[:, feat] <= thresh
    left_subtree = build_tree(X[left_mask], y[left_mask], max_depth, min_samples_leaf, depth + 1)
    right_subtree = build_tree(X[~left_mask], y[~left_mask], max_depth, min_samples_leaf, depth + 1)

    return {
        "leaf": False,
        "feature": feat,
        "threshold": thresh,
        "left": left_subtree,
        "right": right_subtree,
    }

def predict_node(node, x):
    if node["leaf"]:
        return node["class"]
    if x[node["feature"]] <= node["threshold"]:
        return predict_node(node["left"], x)
    else:
        return predict_node(node["right"], x)

def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    return np.array([predict_node(tree, x) for x in X])

 Part 3 How deep should the tree be?

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
depths = [1, 2, 3, 4, 5, 6, 8, 10, None]

results = []
for d in depths:
    model = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=SEED)
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    
    model.fit(X_train, y_train)
    n_leaves = model.get_n_leaves()
    
    results.append({
        "depth": str(d),
        "leaves": n_leaves,
        "train_acc": scores["train_score"].mean(),
        "cv_acc": scores["test_score"].mean(),
        "cv_std": scores["test_score"].std()
    })

df_depths = pd.DataFrame(results)
print(df_depths)

  depth  leaves  train_acc    cv_acc    cv_std
0     1       2   0.756020  0.734146  0.046742
1     2       4   0.764505  0.715215  0.058315
2     3       8   0.859876  0.758769  0.049294
3     4      15   0.893706  0.768177  0.058242
4     5      23   0.940825  0.773171  0.065583
5     6      29   0.964980  0.783043  0.082993
6     8      35   0.996378  0.763531  0.045793
7    10      35   1.000000  0.758769  0.049294
8  None      35   1.000000  0.758769  0.049294


Part 4: Tree vs. Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logreg_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
lr_scores = cross_validate(logreg_pipe, X_train, y_train, cv=cv, scoring=("accuracy", "f1"))

print("LogReg CV Accuracy:", lr_scores["test_accuracy"].mean(), "±", lr_scores["test_accuracy"].std())
print("LogReg CV F1:", lr_scores["test_f1"].mean())

LogReg CV Accuracy: 0.8451800232288036 ± 0.040047966642815916
LogReg CV F1: 0.828288027761712


Part 5: Reading a Tree and Reproducibility